In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

c:\Users\meetu\GenerativeAICampusX\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\meetu\GenerativeAICampusX\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
class CricState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int

    strike_rate: float
    balls_per_boundary: float
    boundary_percentage: float
    summary: str

In [7]:
def calculate_sr(state: CricState) -> CricState:
    strike_rate = round((state["runs"]/state["balls"])*100, 2)
    return {"strike_rate": strike_rate}

def calculate_bpb(state: CricState) -> CricState:
    total_boundaries = state["fours"] + state["sixes"]
    balls_per_boundary = round((total_boundaries/state["balls"])*100, 2)
    return {"balls_per_boundary": balls_per_boundary}

def calculate_bperc(state: CricState) -> CricState:
    four_runs = state["fours"]*4
    six_runs = state["sixes"]*6
    boundary_runs = four_runs + six_runs

    boundary_percentage = round((boundary_runs/state["runs"])*100, 2)
    return {"boundary_percentage": boundary_percentage}

def summary(state: CricState) -> CricState:
    summary = f"""Runs: {state["runs"]}\nStrike Rate: {state["strike_rate"]}\nBalls per boundary: {state["balls_per_boundary"]}\nBoundaries percentage: {state["boundary_percentage"]},
                    """
    return {"summary": summary}

In [8]:
graph = StateGraph(CricState)

graph.add_node("calculate_sr", calculate_sr)
graph.add_node("calculate_bpb", calculate_bpb)
graph.add_node("calculate_bperc", calculate_bperc)
graph.add_node("summary", summary)

graph.add_edge(START, "calculate_sr")
graph.add_edge(START, "calculate_bpb")
graph.add_edge(START, "calculate_bperc")

graph.add_edge("calculate_sr", "summary")
graph.add_edge("calculate_bpb", "summary")
graph.add_edge("calculate_bperc", "summary")

graph.add_edge("summary", END)

workflow = graph.compile()

In [9]:
intitial_state ={"runs": 76, "balls": 31, "fours": 5, "sixes": 4}
final_state = workflow.invoke(intitial_state)["summary"]
print(final_state)

Runs: 76
Strike Rate: 245.16
Balls per boundary: 29.03
Boundaries percentage: 57.89,
                    
